In [ ]:
import polars as pl
import altair as alt
import pyarrow as pa
import matplotlib.pyplot as plt
import seaborn as sns
import vegafusion
import os
from pathlib import Path

alt.data_transformers.enable("vegafusion")
plt.style.use("seaborn-v0_8-whitegrid")

try:
    _base = Path(__file__).resolve().parent.parent
except NameError:
    _base = Path(os.getcwd()).parent
df = pl.read_csv(_base / "data" / "raw" / "edos_labelled_aggregated.csv")
df_pd = df.to_pandas().reset_index()
df_sexist = df.filter(pl.col("label_sexist") == "sexist")
df_sexist_pd = df_sexist.to_pandas().reset_index()

In [ ]:
# Distribution of label_sexist
sexist_counts = df["label_sexist"].value_counts().sort("count", descending=True)
print("=== Label distribution ===")
print(sexist_counts)

In [ ]:
# Distribution of label_category
cat_counts = df["label_category"].value_counts().sort("count", descending=True)
print("\n=== Category distribution ===")
print(cat_counts)

In [ ]:
# Distribution of label_vector
vector_counts = df["label_vector"].value_counts().sort("count", descending=True)
print("\n=== Vector distribution ===")
print(vector_counts)

In [ ]:
# Text length analysis
df_pd["text_length"] = df_pd["text"].str.len()

text_len_by_cat = (
    alt.Chart(df_pd)
    .mark_boxplot()
    .encode(
        x=alt.X("label_category:N", title="Category"),
        y=alt.Y("text_length:Q", title="Text Length (chars)"),
        color=alt.Color("label_category:N", scale=alt.Scale(scheme="rainbow")),
    )
    .properties(title="Text Length by Category", width=500)
)
text_len_by_cat.display()

In [ ]:
# Text length by sexist label
text_len_by_sexist = (
    alt.Chart(df_pd)
    .mark_boxplot()
    .encode(
        x=alt.X("label_sexist:N", title="Sexist"),
        y=alt.Y("text_length:Q", title="Text Length (chars)"),
        color=alt.Color("label_sexist:N", scale=alt.Scale(scheme="rainbow")),
    )
    .properties(title="Text Length by Sexist Label", width=300)
)
text_len_by_sexist.display()

In [ ]:
# Category breakdown within sexist tweets
sexist_only = df_pd[df_pd["label_sexist"] == "sexist"]
cat_pie = (
    alt.Chart(sexist_only)
    .mark_arc()
    .encode(
        theta=alt.Theta("count()", title="Count"),
        color=alt.Color("label_category:N", title="Category"),
    )
    .properties(title="Category Distribution (Sexist Only)")
)
cat_pie.display()

In [ ]:
print(f"\nTotal rows: {len(df)}")
print(f"Sexist: {len(df_sexist)} ({len(df_sexist) / len(df) * 100:.1f}%)")
print(
    f"Not sexist: {len(df) - len(df_sexist)} ({(len(df) - len(df_sexist)) / len(df) * 100:.1f}%)"
)

In [ ]:
"""
Code for saving this file as a notebook.
# It Goes at the end of the file.
"""

In [ ]:
if __name__ == "__main__":
    import jupytext as jpt

    _py = Path(__file__)
    _nb = _py.with_suffix(".ipynb")
    if not _nb.exists():
        jpt.write(jpt.read(_py), _nb, fmt="ipynb")
        print(f"Created {_nb.name}")
    else:
        jpt.write(jpt.read(_py), _nb, fmt="ipynb")
        print(f"Synced {_nb.name}")